# Settimana 6 — Bertrand, Gauss e l’asteroide Cerere: variabili aleatorie continue e distribuzione normale

Questo è il notebook della Settimana 6 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_06/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## Laboratorio: attese simulate

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_06/#lab-esponenziale)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(2026)

LAMBDA = 3.0          # lavori all'ora, in media
N = 100_000           # attese simulate

# Attenzione: NumPy vuole la scala 1/lambda, non il tasso lambda.
attese = rng.exponential(scale=1 / LAMBDA, size=N)
print(f"Attesa media: {attese.mean():.4f} ore   (1/lambda = {1 / LAMBDA:.4f})")

print("  t     P(T > t) simulata   e^(-lambda t)")
for t in [1 / 12, 1 / 3, 1.0]:
    print(f"{t:5.3f}   {np.mean(attese > t):17.4f}   {np.exp(-LAMBDA * t):13.4f}")

# Assenza di memoria: tra le attese già oltre mezz'ora, quante
# superano mezz'ora + 20 minuti? Deve somigliare a P(T > 20 minuti).
lunghe = attese[attese > 0.5]
print(f"P(T > 0,5 + 1/3 | T > 0,5) = {np.mean(lunghe > 0.5 + 1 / 3):.4f}")
print(f"P(T > 1/3)                 = {np.mean(attese > 1 / 3):.4f}")

# Istogramma con altezza = frequenza / larghezza, e la densità teorica.
t = np.linspace(0, 2, 300)
plt.hist(attese, bins=80, range=(0, 2), density=True, alpha=0.6,
         label="Attese simulate")
plt.plot(t, LAMBDA * np.exp(-LAMBDA * t), label="densità λe^(−λt)")
plt.xlabel("Tempo di attesa (ore)")
plt.ylabel("Densità")
plt.legend()
plt.tight_layout()
plt.show()

## Laboratorio: la campana nasce dagli errori

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_06/#lab-normale)

In [ ]:
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1801)   # l'anno di Cerere

# Ogni misura = valore vero + 50 piccoli errori indipendenti,
# ciascuno uniforme tra -0,1 e +0,1 (vibrazioni, lettura, aria...).
N_MISURE, N_ERRORI = 20_000, 50
errori = rng.uniform(-0.1, 0.1, size=(N_MISURE, N_ERRORI))
scarto = errori.sum(axis=1)          # errore totale di ogni misura

mu, sigma = scarto.mean(), scarto.std()
print(f"media degli errori: {mu:+.4f}   sigma: {sigma:.4f}")
for k in [1, 2, 3]:
    dentro = np.mean(np.abs(scarto - mu) < k * sigma)
    print(f"entro {k} sigma: {dentro:.3f}")

# La campana con la stessa media e la stessa sigma.
x = np.linspace(-1.6, 1.6, 400)
campana = [NormalDist(mu, sigma).pdf(v) for v in x]
plt.hist(scarto, bins=60, density=True, alpha=0.6,
         label="Errori simulati")
plt.plot(x, campana, label="Normale con stessa media e sigma")
plt.xlabel("Errore totale di una misura")
plt.ylabel("Densità")
plt.legend()
plt.tight_layout()
plt.show()

# Calcoli con la distribuzione normale.
altezze = NormalDist(mu=170, sigma=10)
redditi = NormalDist(mu=900, sigma=200)
print(f"P(altezza > 180 cm)       = {1 - altezze.cdf(180):.4f}")
print(f"P(600 < reddito < 1200)   = {redditi.cdf(1200) - redditi.cdf(600):.4f}")
print(f"soglia dei buoni (3%)     = {redditi.inv_cdf(0.03):.1f} monete")

## Approfondimento facoltativo — Quando una somma diventa una campana, e quando no

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_06/#lab-normale)

In [ ]:
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1801)
N_MISURE, N_ERRORI = 20_000, 50
FORMA = (N_MISURE, N_ERRORI)   # una riga per misura, una colonna per errore

# Caso 1: ogni errore è uniforme, ma con una larghezza tutta sua.
larghezze = rng.uniform(0.02, 0.2, size=N_ERRORI)
# Caso 2: un errore uniforme tra -1 e 1, gli altri 49 tra -0,01 e 0,01.
dominante = np.array([1.0] + [0.01] * (N_ERRORI - 1))
# Caso 3: lanci di moneta che valgono ±1/2, ±1/4, ±1/8, ...
pesi = 0.5 ** np.arange(1, N_ERRORI + 1)

casi = {
    "larghezze tutte diverse": rng.uniform(-larghezze, larghezze, size=FORMA),
    "un errore grande, 49 minuscoli": rng.uniform(-dominante, dominante,
                                                  size=FORMA),
    "monete che si dimezzano": rng.choice([-1, 1], size=FORMA) * pesi,
}

fig, assi = plt.subplots(1, 3, figsize=(11, 3.2))
for asse, (nome, errori) in zip(assi, casi.items()):
    scarto = errori.sum(axis=1)               # errore totale di ogni misura
    mu, sigma = scarto.mean(), scarto.std()
    varianze = errori.var(axis=0)             # varianza di ciascun errore
    quota = varianze.max() / varianze.sum()   # peso dell'errore più grande
    print(f"{nome}: l'errore più grande porta il {quota:.1%} della varianza")
    dentro = [np.mean(np.abs(scarto - mu) < k * sigma) for k in [1, 2, 3]]
    print(f"   entro 1, 2, 3 sigma: {dentro[0]:.3f}  {dentro[1]:.3f}  "
          f"{dentro[2]:.3f}")
    x = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 400)
    asse.hist(scarto, bins=60, density=True, alpha=0.6)
    asse.plot(x, [NormalDist(mu, sigma).pdf(v) for v in x])
    asse.set_title(nome, fontsize=10)
    asse.set_xlabel("Errore totale")
plt.tight_layout()
plt.show()

## Lettura A — Il paradosso di Bertrand, con i conti

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_06/#bertrand-esteso)

In [ ]:
import random
from math import cos, pi, sin, sqrt

random.seed(1889)            # l'anno del libro di Bertrand
N = 50_000                   # corde per ciascun metodo
LATO = sqrt(3)               # lato del triangolo inscritto (raggio 1)


def corda_estremi():
    """Metodo 1: due punti scelti a caso sulla circonferenza."""
    a = random.uniform(0, 2 * pi)
    b = random.uniform(0, 2 * pi)
    # Distanza tra due punti del cerchio di raggio 1.
    return sqrt((cos(a) - cos(b)) ** 2 + (sin(a) - sin(b)) ** 2)


def corda_raggio():
    """Metodo 2: distanza del punto medio dal centro, uniforme su [0, 1]."""
    d = random.uniform(0, 1)
    return 2 * sqrt(1 - d * d)  # Pitagora: metà corda = sqrt(1 - d^2)


def corda_punto_medio():
    """Metodo 3: punto medio uniforme dentro il disco."""
    while True:
        # Punto uniforme nel quadrato; teniamo solo quelli nel disco.
        x, y = random.uniform(-1, 1), random.uniform(-1, 1)
        if x * x + y * y <= 1:
            return 2 * sqrt(1 - (x * x + y * y))


for nome, metodo, teoria in [("estremi a caso", corda_estremi, 1 / 3),
                             ("raggio a caso", corda_raggio, 1 / 2),
                             ("punto medio a caso", corda_punto_medio,
                              1 / 4)]:
    lunghe = sum(metodo() > LATO for _ in range(N))
    print(f"{nome:20s} frequenza {lunghe / N:.3f}"
          f"   teoria {teoria:.3f}")